# 02. Co-registration and bias

Block 01 ended with a naive DoD that reads about 10 cm high on ground that did not burn, with a color pattern on the hillslopes that follows aspect. In this block we measure that error on stable ground, remove the offset between the two lidar collections with *r.dem.nk*, and check whether pre-fire vegetation leaves a bias worth correcting.

In [ ]:
import io
import json

import matplotlib.pyplot as plt
import numpy as np
from grass.script import array as garray

from workshop_utils import RAW, gj, stable_stats, start

tools, session = start(
    "02", required=["dtm_2013", "dtm_2022", "dod_naive", "burned", "slope_2013"]
)

## Stable ground

Every correction in this block is estimated on ground we expect did not change between 2013 and 2022. Here that means cells that are outside the fire perimeter, sloped between 2 and 45 degrees, and more than 10 m from a channel, since channels carry real deposition and scour.

In [ ]:
tools.r_watershed(elevation="dtm_2022", threshold=10000, stream="streams", flags="a")
tools.r_grow_distance(input="streams", distance="stream_distance")
tools.r_mapcalc(
    expression=(
        "stable = if(burned == 0 && slope_2013 >= 2 && slope_2013 <= 45 "
        "&& stream_distance > 10, 1, null())"
    )
)

tools.r_mapcalc(expression="channels = if(isnull(streams), null(), 1)")
tools.r_colors(map="channels", rules=io.StringIO("1 30:100:255\n"))
tools.r_colors(map="stable", rules=io.StringIO("1 255:200:0\n"))

n_stable = tools.r_univar(map="stable", format="json")["n"]
n_region = tools.g_region(flags="p", format="json")["cells"]
print(f"Stable cells: {n_stable} ({100 * n_stable / n_region:.1f}% of the region)")

In [ ]:
stable_map = gj.Map(width=700)
stable_map.d_rast(map="dtm_2022_relief")
stable_map.d_rast(map="stable")
stable_map.d_rast(map="channels")
stable_map.d_vect(
    map="mckinney_perimeter@PERMANENT", fill_color="none", color="red", width=2
)
stable_map.show()

Now the numbers to beat. The median is the systematic offset and the NMAD (normalized median absolute deviation) is the noise, both measured where the true change should be zero.

In [ ]:
before = stable_stats(tools, "dod_naive", "stable")
print(
    f"Naive DoD on stable ground: median {before['median']:+.3f} m, NMAD {before['nmad']:.3f} m"
)

## Horizontal and vertical offset

A horizontal shift between two DEMs shows up as an elevation error that depends on slope and aspect. *r.dem.nk* fits the Nuth and Kaab (2011) model to that pattern on the stable cells and returns the shift. We align the 2013 DTM to the 2022 reference, since the 2022 collection is the newer, higher-specification survey.

In [ ]:
tools.r_dem_nk(
    sfm="dtm_2013",
    lidar="dtm_2022",
    stable_mask="stable",
    output="dtm_2013_nk",
    transform_output="nk_transform.txt",
)

offsets = {}
with open("nk_transform.txt") as f:
    for line in f:
        if "=" in line:
            key, value = line.strip().split("=")
            offsets[key] = float(value)
horizontal = np.hypot(offsets["dx"], offsets["dy"])
print(
    f"dx {offsets['dx']:+.3f} m, dy {offsets['dy']:+.3f} m, dz {offsets['dz']:+.3f} m"
)
print(f"Horizontal shift: {horizontal:.2f} m")

The horizontal shift is about 5 cm, a twentieth of a cell. In a nearest-cell DoD the same fit finds about 0.35 m, almost all of it the grid offset that block 01 resampled away. The vertical offset of about 9 cm is larger than the 2 to 3 cm geoid difference from block 01, so the geoid change explains only part of it. An iterative closest point fit with *r.dem.icp* on the same mask points the same way, with about 8 cm horizontal and 10 cm vertical. The two methods differ by about 3 cm, close to the size of the horizontal shift itself, so the vertical offset is well resolved and the horizontal one is small and uncertain.

The small shift also settles the datum question from block 01. If the 2013 data were on the original NAD83, the fit would find a shift near 0.6 m. It finds 5 cm, so applying the NADCON5 grid shift would have added error, not removed it.

Difference the aligned 2013 DTM against 2022 with *r.dem.dod*. Both DoDs use `range=1` so they share one color scale.

In [ ]:
tools.r_dem_dod(dem="dtm_2022", reference="dtm_2013", output="dod_naive", range=1)
tools.r_dem_dod(dem="dtm_2022", reference="dtm_2013_nk", output="dod_nk", range=1)

after = stable_stats(tools, "dod_nk", "stable")
print(f"{'':12}{'median':>10}{'NMAD':>10}")
for label, stats in (("naive", before), ("r.dem.nk", after)):
    print(f"{label:12}{stats['median']:>+10.3f}{stats['nmad']:>10.3f}")

In [ ]:
for name in ("dod_naive", "dod_nk"):
    dod_map = gj.Map(width=500)
    dod_map.d_shade(shade="dtm_2022_relief", color=name)
    dod_map.d_vect(
        map="mckinney_perimeter@PERMANENT", fill_color="none", color="black", width=2
    )
    dod_map.d_legend(
        raster=name, title=f"{name} (m)", range=(-1, 1), at=(5, 40, 2, 5), flags="b"
    )
    dod_map.show()

The median drops from about +0.10 to +0.01 m, while the NMAD barely moves. A vertical offset moves every cell by the same amount, and the horizontal shift is small, so removing them recenters the error without narrowing it.

To check the horizontal part, plot the stable-ground error divided by the tangent of slope against aspect. A horizontal shift traces a cosine in this plot, and a vertical offset lifts the whole curve.

In [ ]:
tools.r_slope_aspect(
    elevation="dtm_2022", slope="slope_2022", aspect="aspect_2022", flags="n"
)
slope = garray.array("slope_2022", null=np.nan)
aspect = garray.array("aspect_2022", null=np.nan)
# Read the integer mask as float so null cells become NaN.
stable_cells = garray.array("stable", dtype=np.float64, null=np.nan)
on_stable = ~np.isnan(stable_cells) & (slope >= 5)

edges = np.arange(0, 361, 15)
centers = edges[:-1] + 7.5
fig, ax = plt.subplots(figsize=(7, 3.5))
for name, label in (("dod_naive", "naive"), ("dod_nk", "after r.dem.nk")):
    ratio = garray.array(name, null=np.nan) / np.tan(np.radians(slope))
    medians = [
        np.nanmedian(ratio[on_stable & (aspect >= lo) & (aspect < hi)])
        for lo, hi in zip(edges[:-1], edges[1:])
    ]
    ax.plot(centers, medians, marker="o", label=label)
ax.axhline(0, color="gray", linewidth=0.8)
ax.set_xlabel("Aspect (degrees from north)")
ax.set_ylabel("DoD / tan(slope) (m)")
ax.set_xticks(range(0, 361, 45))
ax.legend()
plt.show()

Before co-registration the curve sits high, which is the vertical offset, with a gentle cosine on top. After *r.dem.nk* it sits near zero with a few centimeters of wiggle left.

## Vegetation

Pre-fire canopy can bias a DTM. Dense canopy leaves fewer ground returns, so the 2013 bare-earth surface is interpolated over larger gaps than the 2022 surface flown after the fire. The 2013 highest-hit surface minus the bare-earth surface gives canopy height. A few highest-hit returns are blunders hundreds of meters up, so heights above 70 m are dropped.

In [ ]:
tools.r_import(
    input=str(RAW / "usfs_2013_dsm.tif"),
    output="dsm_2013",
    extent="region",
    resolution="region",
    resample="bilinear",
)
tools.r_mapcalc(
    expression=(
        "canopy_2013 = if(dsm_2013 - dtm_2013 > 70, null(), "
        "max(dsm_2013 - dtm_2013, 0))"
    )
)
tools.r_mapcalc(expression="forest = if(canopy_2013 > 2, 1, null())")
tools.r_mapcalc(
    expression="stable_forest = if(!isnull(stable) && !isnull(forest), 1, null())"
)
tools.r_mapcalc(
    expression="stable_open = if(!isnull(stable) && isnull(forest), 1, null())"
)
tools.r_mapcalc(
    expression="burned_forest = if(burned == 1 && !isnull(forest), 1, null())"
)

for label, mask in (
    ("stable forest", "stable_forest"),
    ("stable open", "stable_open"),
    ("burned forest", "burned_forest"),
):
    stats = stable_stats(tools, "dod_nk", mask)
    print(
        f"{label:14} median {stats['median']:+.3f} m, NMAD {stats['nmad']:.3f} m, cells {stats['n']}"
    )

On stable ground, forest and open cells differ by about 2 cm. Burned forest reads about 12 cm lower, which is change we want to keep.

*r.dem.bias* `method=forest` estimates a local median under a canopy mask and subtracts it. Applied to burned forest, it would remove that 12 cm along with any bias, because a local median cannot tell the two apart. `method=regression` instead fits the bias to terrain and canopy predictors on stable ground only and predicts it everywhere, which keeps real change out of the fit.

In [ ]:
tools.r_dem_bias(
    dod="dod_nk",
    output="dod_debiased",
    method="regression",
    predictors="slope_2013,canopy_2013",
    stable_mask="stable",
    bias_field="bias_regression",
    fit_json="bias_fit.json",
)

with open("bias_fit.json") as f:
    fit = json.load(f)
print("Intercept:", round(fit["intercept"], 4))
for name, value in fit["coefficients"].items():
    print(f"{name:12} {value:+.4f} m per standard deviation")

debiased = stable_stats(tools, "dod_debiased", "stable")
print(
    f"Stable ground after regression: median {debiased['median']:+.3f} m, NMAD {debiased['nmad']:.3f} m"
)

The regression changes the stable-ground median by millimeters. After co-registration there is little vegetation bias left to remove at this site, and measuring it first is how you know. Block 03 starts from `dod_debiased`.

## Your turn

The stable mask decides what *r.dem.nk* sees. Rebuild it without the burn and channel conditions, rerun *r.dem.nk*, and compare the offsets with the ones above. Change and channels pull the fit toward the real surface change.

Then solve the same alignment with *r.dem.icp*, which matches the surfaces point to plane instead of fitting the aspect pattern. Its signs are reversed because it reports the transform applied to the 2013 DTM.

In [ ]:
tools.r_mapcalc(
    expression="stable_loose = if(slope_2013 >= 2 && slope_2013 <= 45, 1, null())"
)
tools.r_dem_nk(
    sfm="dtm_2013",
    lidar="dtm_2022",
    stable_mask="stable_loose",
    output="dtm_2013_nk_loose",
    transform_output="nk_transform_loose.txt",
)
print(open("nk_transform_loose.txt").read())

In [ ]:
tools.r_dem_icp(
    reference="dtm_2022",
    source="dtm_2013",
    output="dtm_2013_icp",
    mask="stable",
    dof=4,
    transform_out="icp_transform.txt",
)
print(open("icp_transform.txt").read())